In [ ]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import poisson
import numpy as np
import joblib
from datetime import datetime, timedelta
import random
import os

In [ ]:
df = pd.read_csv("road_accident.csv", encoding='latin1')

In [ ]:
df['DATE ENCODED'] = pd.to_datetime(df['DATE ENCODED'], errors='coerce')
df = df.dropna(subset=['DATE ENCODED'])

In [ ]:
daily_incidents = df.groupby(df['DATE ENCODED'].dt.date).size()
daily_incidents.index = pd.to_datetime(daily_incidents.index)
daily_incidents = daily_incidents.sort_index()


In [ ]:
print(f"Data range: {daily_incidents.index[0].date()} → {daily_incidents.index[-1].date()}")

Data range: 2022-01-01 → 2023-12-01


In [ ]:
total_size = len(daily_incidents)
train_size = int(total_size * 0.8)

train = daily_incidents[:train_size]
test = daily_incidents[train_size:]

In [ ]:
print("\nTraining ARIMA model...")
model = ARIMA(daily_incidents, order=(2,1,2))
model_fit = model.fit()


Training ARIMA model...


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


In [ ]:
model = ARIMA(train, order=(2,1,2))
model_fit = model.fit()
forecast = model_fit.forecast(steps=len(test))

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/p

In [ ]:
# CORRECT METRICS FOR ARIMA
mae = mean_absolute_error(test, forecast)
rmse = np.sqrt(mean_squared_error(test, forecast))
mape = np.mean(np.abs((test.values - forecast) / test.values)) * 100

In [ ]:
print(f"MAE: {mae:.2f} accidents")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.1f}% error")

MAE: 0.87 accidents
RMSE: 1.02
MAPE: 64.7% error


In [ ]:
# === 3. Retrain on FULL data for best 2023 forecast ===
print("\nRetraining on full 2022 data for 2023 forecast...")
final_model = ARIMA(daily_incidents, order=(2,1,2))
final_fit = final_model.fit()


Retraining on full 2022 data for 2023 forecast...


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


In [ ]:
# === 4. Forecast daily for all of 2023 ===
last_date = daily_incidents.index[-1]
days_to_2023_end = (datetime(2023, 12, 31) - last_date).days + 1  # include last day

In [ ]:
print(f"Forecasting {days_to_2023_end} days into 2023...")

Forecasting 31 days into 2023...


In [ ]:
forecast_daily = final_fit.forecast(steps=days_to_2023_end)

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [ ]:
# Create date range for 2023
forecast_dates = pd.date_range(start=last_date + timedelta(days=1), periods=days_to_2023_end, freq='D')
forecast_series = pd.Series(forecast_daily, index=forecast_dates)

In [ ]:
# === 5. Aggregate to MONTHLY 2023 totals ===
monthly_2023 = forecast_series.resample('ME').sum()

In [ ]:
# Round to realistic whole numbers
monthly_2023 = monthly_2023.round().astype(int)

In [ ]:
print("\n" + "="*60)
print("       2023 MONTHLY ROAD ACCIDENT FORECAST")
print("="*60)
for month, count in monthly_2023.items():
    print(f"{month.strftime('%B %Y')}: {count} accidents")

print("="*60)


       2023 MONTHLY ROAD ACCIDENT FORECAST
December 2023: 0 accidents
January 2024: 0 accidents


In [ ]:
joblib.dump(model_fit, '/content/monthly_arima80_20.pkl')
print("Model saved: /content/monthly_arima80_20.pkl")

Model saved: /content/monthly_arima80_20.pkl


In [ ]:
# Optional: Save monthly forecast to CSV for reference
monthly_2023.to_csv('2023_monthly_forecast.csv')
print("Monthly forecast saved to '2023_monthly_forecast.csv'")

Monthly forecast saved to '2023_monthly_forecast.csv'


In [ ]:
# === 3. Generate RANDOM, DYNAMIC 2023 Prediction ===
def predict_2023_risk():
    # Base forecast from ARIMA
    last_date = daily_incidents.index[-1]
    days_to_end_2023 = max(1, (datetime(2023, 12, 31) - last_date).days)

    # Forecast into 2023
    base_forecast = model_fit.forecast(steps=days_to_end_2023 + 180)
    predicted_2023_total = float(base_forecast.sum())
    predicted_2023_total = max(50, predicted_2023_total)  # minimum realistic

    # === RANDOM VARIATION FOR MOVEMENT ===
    # Add natural randomness so % changes every time
    random_boost = random.uniform(0.15, 0.45)  # 15-45% extra variation
    random_noise = random.uniform(-8, 12)      # small swing

    adjusted_total = predicted_2023_total * (1 + random_boost) + random_noise
    adjusted_total = max(60, min(180, adjusted_total))  # keep realistic range

    # Convert to % risk (0-98%)
    base_prob = (adjusted_total / 150) * 100  # scale to max ~150 accidents/year
    final_prob = min(98.9, max(20.0, base_prob + random.uniform(-5, 8)))

    # Final message — clean and professional
    prediction_text = f"In 2023, there is a {final_prob:.1f}% chance another road accident will occur"

    return prediction_text, final_prob, adjusted_total

# === 4. Run prediction (test) ===
prediction, prob, count = predict_2023_risk()

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [ ]:
print("\n" + "="*70)
print("           2023 ROAD ACCIDENT RISK FORECAST (DYNAMIC)")
print("="*70)
print(f"Forecasted accidents in 2023: ~{int(count)}")
print(f"Risk probability: {prob:.1f}%")
print("\nFINAL PREDICTION MESSAGE:")
print(prediction)
print("="*70)


           2023 ROAD ACCIDENT RISK FORECAST (DYNAMIC)
Forecasted accidents in 2023: ~180
Risk probability: 98.9%

FINAL PREDICTION MESSAGE:
In 2023, there is a 98.9% chance another road accident will occur


In [ ]:
with open("current_road_risk.txt", "w") as f:
    f.write(prediction)

# Task
Prepare training data for the ARIMA model by filtering 'daily_incidents' to include data up to December 31, 2022. Then, retrain the ARIMA model (order 2,1,2) with this data. Forecast daily road incidents for January 1, 2023, to December 31, 2023, and extract the forecast for July 1, 2023, to December 31, 2023. Sum these forecasted incidents to use as the lambda parameter for a Poisson distribution, calculating the probability of 8 or more road accidents in this six-month period. Finally, print the calculated probability in a user-friendly format as the final prediction message.

## Prepare Training Data for 2022

### Subtask:
Filter the 'daily_incidents' DataFrame to include only data up to December 31, 2022. This subset will be used as the training data for the ARIMA model.


**Reasoning**:
To prepare the training data for the ARIMA model, I will filter the 'daily_incidents' DataFrame to include only the data up to December 31, 2022, and store it in a new variable.



In [ ]:
daily_incidents_2022 = daily_incidents[daily_incidents.index <= '2022-12-31']
print(f"Data range for 2022 incidents: {daily_incidents_2022.index.min().date()} → {daily_incidents_2022.index.max().date()}")
print(f"Number of incidents in 2022 data: {len(daily_incidents_2022)}")

NameError: name 'daily_incidents' is not defined

**Reasoning**:
The previous code failed with a `NameError` because 'daily_incidents' was not defined in the current execution context. Although `daily_incidents` was defined in an earlier successful cell, the current environment does not retain its value. To fix this, I need to include the necessary preceding steps to define `daily_incidents` within this code block, starting from loading the data and preprocessing the date column.



In [ ]:
import pandas as pd

df = pd.read_csv("road_accident.csv", encoding='latin1')
df['DATE ENCODED'] = pd.to_datetime(df['DATE ENCODED'], errors='coerce')
df = df.dropna(subset=['DATE ENCODED'])

daily_incidents = df.groupby(df['DATE ENCODED'].dt.date).size()
daily_incidents.index = pd.to_datetime(daily_incidents.index)
daily_incidents = daily_incidents.sort_index()

daily_incidents_2022 = daily_incidents[daily_incidents.index <= '2022-12-31']
print(f"Data range for 2022 incidents: {daily_incidents_2022.index.min().date()} → {daily_incidents_2022.index.max().date()}")
print(f"Number of incidents in 2022 data: {len(daily_incidents_2022)}")

Data range for 2022 incidents: 2022-01-01 → 2022-12-06
Number of incidents in 2022 data: 75


## Train ARIMA Model on 2022 Data

### Subtask:
Retrain the ARIMA model with the order (2,1,2) using the prepared training data that spans until the end of 2022.


**Reasoning**:
I will retrain the ARIMA model using the `daily_incidents_2022` data, which contains incident data up to December 31, 2022, with the specified order (2,1,2).



In [ ]:
from statsmodels.tsa.arima.model import ARIMA

model_2022 = ARIMA(daily_incidents_2022, order=(2,1,2))
model_fit_2022 = model_2022.fit()
print("ARIMA model trained successfully on 2022 data.")

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


ARIMA model trained successfully on 2022 data.


**Reasoning**:
The ARIMA model has been successfully trained on 2022 data. The next logical step is to forecast daily road incidents for the entire year 2023 as specified in the task. I will calculate the number of forecast steps needed from the end of the training data to the end of 2023, generate the forecast, create a proper date index for the forecasted values, and then extract the relevant period from July 1, 2023, to December 31, 2023.



In [ ]:
from datetime import datetime, timedelta
from scipy.stats import poisson
import numpy as np
import pandas as pd

# Define the forecast period for the entire year 2023
forecast_start_date_full_year = daily_incidents_2022.index.max() + timedelta(days=1)
forecast_end_date_full_year = datetime(2023, 12, 31)

# Calculate the number of steps needed for forecasting from the end of training data to the end of 2023
num_forecast_steps = (forecast_end_date_full_year - daily_incidents_2022.index.max()).days

# Generate the forecast for the entire period up to the end of 2023
forecast_2023_full = model_fit_2022.forecast(steps=num_forecast_steps)

# Create a date index for the forecast results
forecast_dates_full = pd.date_range(start=forecast_start_date_full_year, end=forecast_end_date_full_year, freq='D')
forecast_series_full = pd.Series(forecast_2023_full, index=forecast_dates_full)

# Extract the forecast for July 1, 2023, to December 31, 2023
july_to_dec_forecast = forecast_series_full['2023-07-01':'2023-12-31']

# Sum the forecasted incidents for this period to use as lambda for Poisson distribution
lambda_param = july_to_dec_forecast.sum()

# Ensure lambda_param is non-negative, as Poisson distribution requires lambda >= 0
lambda_param = max(0, lambda_param)

# Calculate the probability of 8 or more road accidents using Poisson distribution
prob_high_risk = 1 - poisson.cdf(7, lambda_param)

# Convert to percentage and format the final prediction message
final_prob_percentage = prob_high_risk * 100
final_prediction = f"There is a {final_prob_percentage:.1f}% chance of 8 or more road accidents between July 1, 2023, and December 31, 2023."

print(final_prediction)

There is a 0.0% chance of 8 or more road accidents between July 1, 2023, and December 31, 2023.


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
